In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from utils import twh_to_ej_str, build_const_value_xml, build_const_techs_xml, write_text, twh_to_ej, xy, gw_to_twh
from pathlib import Path

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * the Monthly Report on Major Electric Power Statistics (No. 565, November 2025) (MES, `../resources/MES-565-20260119`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)

* Implemented Input Files
    * `/input/policy/korea-2035/power/nuclear_const_value.xml`
    * `/input/policy/korea-2035/power/nuclear_const_techs.xml`

# Nuclear

Nuclear power generation (TWh) is projected for 2030 and 2035 in the BPESD.
For 2020, the floor is set using historical data (ES).
For 2025, the floor is based on observed nuclear generation from December 2024 to November 2025 (MES).
The resulting projections are shown in the figure below.

In [2]:
dictCapTWh = {2020: 160.18, 2021: 158.0, 2022: 176.1, 2023: 180.5, 2024: 188.8, 2025: 186.0, 2030: 204.2, 2035: 236.0}

In [4]:
years_cap, values_cap = xy(dictCapTWh)

fig = go.Figure()

for name, x, y, dash in [
    ("Current Policies", years_cap, values_cap, None),
]:
    fig.add_trace(go.Scatter(
        x=list(x), y=list(y),
        mode='lines+markers',
        name=name,
        line=(dict(dash=dash) if dash else None)
    ))

# Build annotations without repeating blocks
target_years = [2020, 2025, 2030, 2035]
annotations = []
for yr in target_years:
    val = dictCapTWh.get(yr)
    if val is not None:
        annotations.append(go.layout.Annotation(
            x=yr, y=val,
            xanchor='center', yanchor='bottom',
            text=f"{val:.1f} TWh",
            showarrow=True, arrowhead=1, ax=0, ay=-20,
            font=dict(size=15),
        ))

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    width=800, height=600,
    annotations=annotations,
    xaxis=dict(
        # title='Year',
        title_font=dict(size=18),
        tickfont=dict(size=15),
        tickvals=[2020, 2025, 2030, 2035],  # custom tick positions
        range=[2018, 2036]                  # xrange
    ),
    yaxis=dict(
        title='TWh',
        title_font=dict(size=18),
        tickfont=dict(size=15),
        range=[50, 270]                       # yrange
    ),
)

import plotly.io as pio
pio.write_image(fig, "../figure/nuclear.jpg", width=800, height=600, scale=3)
fig.show()

In [10]:
years = [2020, 2025, 2030, 2035]
values_by_year = {y: twh_to_ej_str(dictCapTWh[y]) for y in years}
policy_name = "Nuclear-Ceiling"

xml_value = build_const_value_xml(
    values_by_year=values_by_year,
    policy_name="Nuclear-Ceiling",
    policy_type="tax",
    min_price=True,
    min_price_values_by_year={2020:0, 2025: -1, 2030: -1, 2035: -1}
)
xml_techs = build_const_techs_xml(
    years=[2020, 2025, 2030, 2035],
    sector_name="electricity",
    subsector_name="nuclear",
    policy_name="Nuclear-Ceiling",
    tech_names=["Gen_II_LWR", "Gen_III"],
    policy_type='tax',
)

value_path = "../../input/policy/korea-2035/power/nuclear_const_value.xml"
techs_path = "../../input/policy/korea-2035/power/nuclear_const_techs.xml"

write_text(value_path, xml_value)
write_text(techs_path, xml_techs)

print("Wrote:", Path(value_path).expanduser())
print("Wrote:", Path(techs_path).expanduser())


Wrote: ../../input/policy/korea-2035/power/nuclear_const_value.xml
Wrote: ../../input/policy/korea-2035/power/nuclear_const_techs.xml
